# Model Collapse Simulation — distilgpt2 on TinyStories

Recursive-generation experiment following Shumailov et al. (2024), *"AI models collapse when trained on recursively generated data"* (and the earlier *"Curse of Recursion"* preprint).

**Loop:**

1. Finetune `distilgpt2` on a small slice of [`roneneldan/TinyStories`](https://huggingface.co/datasets/roneneldan/TinyStories) → $M_0$ (real-data baseline; intentionally a different domain than WikiText-2).
2. For $i = 1, \ldots, N$:
   - Sample $D_i$ by generating from $M_{i-1}$ (`top_p=0.9`, `T=1.0`).
   - $M_i = \text{continue\_finetune}(M_{i-1}, D_i)$ — fully synthetic loop, no real data mixed in.
3. After every generation log:
   - **perplexity on a fixed real-data held-out set** — the canonical collapse signal (should *increase*),
   - **token entropy / vocab size / distinct-1 / distinct-2 / mean length** of $D_i$ — diversity signals (should *decrease*).
4. Every checkpoint + a 20-sample preview of every $D_i$ is written to `/kaggle/working/` and zipped at the end.

Sized to finish on a Kaggle T4 in about 90–120 minutes. Adjust `CFG['n_generations']` / `CFG['n_synthetic_per_gen']` if you want longer runs.

## 1. Dependencies

Kaggle's base image already ships transformers/datasets/torch. We just pin to versions known to behave well together for this experiment.

In [ ]:
!pip install -q --upgrade "transformers>=4.40,<5" "datasets>=2.18" "accelerate>=0.30" "evaluate>=0.4"


## 2. Imports, seeds, configuration

In [ ]:
import gc
import json
import math
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


In [ ]:
CFG = dict(
    base_model="distilgpt2",
    dataset_name="roneneldan/TinyStories",

    n_generations=5,

    n_real_train=8000,
    n_heldout=500,
    n_synthetic_per_gen=5000,

    block_size=128,
    train_batch_size=8,
    gen_batch_size=16,
    initial_train_epochs=2,
    n_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    fp16=True,

    gen_max_new_tokens=128,
    gen_top_p=0.9,
    gen_temperature=1.0,
    prompt_token_count=8,       
)

BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "outputs"
BASE.mkdir(parents=True, exist_ok=True)
MODELS_DIR = BASE / "models"
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR = BASE / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
SAMPLES_DIR = BASE / "samples"
SAMPLES_DIR.mkdir(exist_ok=True)

METRICS_PATH = BASE / "collapse_metrics.csv"
CONFIG_PATH = BASE / "config.json"
ZIP_PATH = BASE / "collapse_models.zip"

CONFIG_PATH.write_text(json.dumps(CFG, indent=2))
print("output dir:", BASE)
print("config saved to", CONFIG_PATH)


## 3. Load tokenizer + base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(CFG["base_model"])
base_model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in base_model.parameters()) / 1e6
print(f"vocab size: {tokenizer.vocab_size}")
print(f"params: {n_params:.1f} M")


## 4. Load and split TinyStories

In [ ]:
print("loading TinyStories...")
ds = load_dataset(CFG["dataset_name"], split="train")
total = len(ds)
print(f"  full train split has {total:,} rows")

rng = np.random.default_rng(SEED)
needed = CFG["n_real_train"] + CFG["n_heldout"]
indices = rng.choice(total, size=needed, replace=False)
indices = [int(i) for i in indices]

real_train_rows = ds.select(indices[: CFG["n_real_train"]])
real_heldout_rows = ds.select(indices[CFG["n_real_train"]:])

real_texts = [r["text"] for r in real_train_rows]
heldout_texts = [r["text"] for r in real_heldout_rows]

print(f"real_train: {len(real_texts)}  heldout: {len(heldout_texts)}")
print("\nfirst real example:")
print(real_texts[0][:300])


## 5. Helpers

In [ ]:
def tokenize_to_blocks(texts: List[str], block_size: int) -> Dataset:
    """Concatenate `texts` into one long token stream then chunk into fixed blocks."""
    enc = tokenizer(texts, truncation=False, return_attention_mask=False)
    all_ids: List[int] = []
    for ids in enc["input_ids"]:
        all_ids.extend(ids)
        all_ids.append(tokenizer.eos_token_id)
    n_blocks = len(all_ids) // block_size
    if n_blocks == 0:
        raise ValueError("Not enough tokens to form a single block")
    blocks = [all_ids[i * block_size:(i + 1) * block_size] for i in range(n_blocks)]
    return Dataset.from_dict({"input_ids": blocks})


In [ ]:
@torch.no_grad()
def evaluate_perplexity(model, texts: List[str], block_size: int, batch_size: int = 4) -> float:
    """Token-level perplexity: average shifted cross-entropy over fixed-length blocks."""
    model.eval().to(DEVICE)
    enc = tokenizer(texts, truncation=False, return_attention_mask=False)
    all_ids: List[int] = []
    for ids in enc["input_ids"]:
        all_ids.extend(ids)
        all_ids.append(tokenizer.eos_token_id)
    n_blocks = len(all_ids) // block_size
    if n_blocks == 0:
        return float("nan")
    blocks = torch.tensor(
        [all_ids[i * block_size:(i + 1) * block_size] for i in range(n_blocks)],
        dtype=torch.long,
        device=DEVICE,
    )
    total_loss = 0.0
    total_tokens = 0
    for i in range(0, len(blocks), batch_size):
        batch = blocks[i:i + batch_size]
        out = model(input_ids=batch, labels=batch)
        n_tok = batch.shape[0] * (batch.shape[1] - 1)
        total_loss += out.loss.item() * n_tok
        total_tokens += n_tok
    return math.exp(total_loss / total_tokens)


In [ ]:
def compute_diversity(texts: List[str]) -> Dict[str, float]:
    """Lightweight collapse signals on a corpus."""
    if not texts:
        return {"vocab_size": 0, "entropy": 0.0, "distinct_1": 0.0,
                "distinct_2": 0.0, "mean_length": 0.0, "n_samples": 0, "total_tokens": 0}

    per_text = [tokenizer.encode(t, add_special_tokens=False) for t in texts]
    flat = [t for toks in per_text for t in toks]
    if not flat:
        return {"vocab_size": 0, "entropy": 0.0, "distinct_1": 0.0,
                "distinct_2": 0.0, "mean_length": 0.0,
                "n_samples": len(texts), "total_tokens": 0}

    counts = Counter(flat)
    total = len(flat)
    probs = np.fromiter(counts.values(), dtype=np.float64) / total
    entropy = float(-np.sum(probs * np.log2(probs + 1e-12)))
    bigrams = [(flat[i], flat[i + 1]) for i in range(len(flat) - 1)]
    return {
        "vocab_size": len(counts),
        "entropy": entropy,
        "distinct_1": len(set(flat)) / total,
        "distinct_2": (len(set(bigrams)) / len(bigrams)) if bigrams else 0.0,
        "mean_length": float(np.mean([len(t) for t in per_text])),
        "n_samples": len(texts),
        "total_tokens": total,
    }


In [ ]:
@torch.no_grad()
def generate_synthetic_corpus(model, prompt_pool: List[str], n_samples: int,
                              batch_size: int = 16, verbose_every: int = 10) -> List[str]:
    """Sample n_samples prompts from `prompt_pool`, generate continuations, return list of texts."""
    model.eval().to(DEVICE)

    if len(prompt_pool) >= n_samples:
        prompts = random.sample(prompt_pool, n_samples)
    else:
        prompts = random.choices(prompt_pool, k=n_samples)

    pad_id = tokenizer.pad_token_id
    eos_id = tokenizer.eos_token_id
    prompt_ids: List[List[int]] = []
    for p in prompts:
        ids = tokenizer.encode(p, add_special_tokens=False)
        if len(ids) < CFG["prompt_token_count"]:
            ids = ids + [eos_id] * (CFG["prompt_token_count"] - len(ids))
        prompt_ids.append(ids[: CFG["prompt_token_count"]])

    outputs: List[str] = []
    n_batches = (n_samples + batch_size - 1) // batch_size
    for bi, start in enumerate(range(0, n_samples, batch_size)):
        batch = prompt_ids[start:start + batch_size]
        ids = torch.tensor(batch, dtype=torch.long, device=DEVICE)
        attn = torch.ones_like(ids)
        gen = model.generate(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=CFG["gen_max_new_tokens"],
            do_sample=True,
            top_p=CFG["gen_top_p"],
            temperature=CFG["gen_temperature"],
            pad_token_id=pad_id,
        )
        outputs.extend(tokenizer.batch_decode(gen, skip_special_tokens=True))
        if verbose_every and (bi + 1) % verbose_every == 0:
            print(f"    batch {bi + 1}/{n_batches}", flush=True)
    return outputs


In [ ]:
def train_one_generation(model, train_texts: List[str], output_dir: Path, epochs: int):
    """Continue-train `model` on `train_texts` and save the resulting checkpoint."""
    train_ds = tokenize_to_blocks(train_texts, CFG["block_size"])
    collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=str(output_dir / "_trainer"),
        per_device_train_batch_size=CFG["train_batch_size"],
        num_train_epochs=epochs,
        learning_rate=CFG["learning_rate"],
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        logging_steps=100,
        save_strategy="no",
        report_to=[],
        fp16=CFG["fp16"] and torch.cuda.is_available(),
        remove_unused_columns=False,
        seed=SEED,
        dataloader_pin_memory=False,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        data_collator=collator,
    )
    trainer.train()

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    trainer_tmp = output_dir / "_trainer"
    if trainer_tmp.exists():
        shutil.rmtree(trainer_tmp, ignore_errors=True)
    return model


## 6. $M_0$ — finetune distilgpt2 on the real subset

In [ ]:
m0_dir = MODELS_DIR / "gen_0"
m0_dir.mkdir(exist_ok=True)
model = base_model.to(DEVICE)
t0 = time.time()
model = train_one_generation(model, real_texts, m0_dir, CFG["initial_train_epochs"])
print(f"M_0 training took {time.time() - t0:.1f}s")
print(f"saved to {m0_dir}")


In [ ]:
ppl_0 = evaluate_perplexity(model, heldout_texts, CFG["block_size"])
preview_0 = generate_synthetic_corpus(model, real_texts, n_samples=8, batch_size=8, verbose_every=0)
div_preview_0 = compute_diversity(preview_0)

print(f"M_0 perplexity on real heldout: {ppl_0:.3f}")
print(f"M_0 preview diversity: {div_preview_0}")
print("\nSample from M_0:")
print(preview_0[0][:400])


## 7. Recursive collapse loop

For $i = 1 \ldots N$:

1. Generate $D_i$ from $M_{i-1}$.
2. Train $M_i = \text{continue}(M_{i-1}, D_i)$.
3. Measure collapse: perplexity on **real** held-out, and diversity statistics on $D_i$.
4. Save the checkpoint and a 20-sample preview of $D_i$.

After this cell finishes, `metrics_df` contains one row per generation.

In [ ]:
metrics_log = [{
    "generation": 0,
    "perplexity_real_heldout": ppl_0,
    "trained_on": "real_data",
    "corpus_vocab_size": div_preview_0["vocab_size"],
    "corpus_entropy": div_preview_0["entropy"],
    "corpus_distinct_1": div_preview_0["distinct_1"],
    "corpus_distinct_2": div_preview_0["distinct_2"],
    "corpus_mean_length": div_preview_0["mean_length"],
    "corpus_n_samples": div_preview_0["n_samples"],
    "corpus_total_tokens": div_preview_0["total_tokens"],
}]

prompt_pool = real_texts

for gen_idx in range(1, CFG["n_generations"] + 1):
    print(f"Generation {gen_idx}")

    print(f"  generating {CFG['n_synthetic_per_gen']} samples from M_{gen_idx - 1}...")
    t0 = time.time()
    synthetic_corpus = generate_synthetic_corpus(
        model,
        prompt_pool,
        n_samples=CFG["n_synthetic_per_gen"],
        batch_size=CFG["gen_batch_size"],
    )
    gen_time = time.time() - t0
    print(f"  generation took {gen_time:.1f}s")

    div_corpus = compute_diversity(synthetic_corpus)
    print(f"  D_{gen_idx} -> vocab={div_corpus['vocab_size']}  "
          f"entropy={div_corpus['entropy']:.3f}  "
          f"distinct_1={div_corpus['distinct_1']:.4f}  "
          f"distinct_2={div_corpus['distinct_2']:.4f}  "
          f"mean_len={div_corpus['mean_length']:.1f}")

    preview_path = SAMPLES_DIR / f"D_{gen_idx}.txt"
    preview_path.write_text("\n\n--- next sample ---\n\n".join(synthetic_corpus[:20]),
                            encoding="utf-8")

    print(f"  training M_{gen_idx} on D_{gen_idx}...")
    t0 = time.time()
    m_dir = MODELS_DIR / f"gen_{gen_idx}"
    m_dir.mkdir(exist_ok=True)
    model = train_one_generation(model, synthetic_corpus, m_dir, CFG["n_train_epochs"])
    train_time = time.time() - t0
    print(f"  training took {train_time:.1f}s")

    ppl_i = evaluate_perplexity(model, heldout_texts, CFG["block_size"])
    print(f"  M_{gen_idx} perplexity on real heldout: {ppl_i:.3f}")

    qual = generate_synthetic_corpus(model, real_texts, n_samples=4, batch_size=4, verbose_every=0)
    print(f"  M_{gen_idx} sample:\n  {qual[0][:300]}")

    metrics_log.append({
        "generation": gen_idx,
        "perplexity_real_heldout": ppl_i,
        "trained_on": f"D_{gen_idx} (synthetic from M_{gen_idx - 1})",
        "corpus_vocab_size": div_corpus["vocab_size"],
        "corpus_entropy": div_corpus["entropy"],
        "corpus_distinct_1": div_corpus["distinct_1"],
        "corpus_distinct_2": div_corpus["distinct_2"],
        "corpus_mean_length": div_corpus["mean_length"],
        "corpus_n_samples": div_corpus["n_samples"],
        "corpus_total_tokens": div_corpus["total_tokens"],
        "gen_time_s": gen_time,
        "train_time_s": train_time,
    })

    pd.DataFrame(metrics_log).to_csv(METRICS_PATH, index=False)

    prompt_pool = synthetic_corpus

    del synthetic_corpus
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_df = pd.DataFrame(metrics_log)
print("\nCollapse trajectory")
print(metrics_df.to_string(index=False))


## 8. Plot the collapse trajectory

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(metrics_df["generation"], metrics_df["perplexity_real_heldout"],
                "o-", lw=2)
axes[0, 0].set_title("Perplexity on real held-out")
axes[0, 0].set_xlabel("generation")
axes[0, 0].set_ylabel("perplexity")
axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(metrics_df["generation"], metrics_df["corpus_vocab_size"],
                "o-", lw=2)
axes[0, 1].set_title("Vocab size of $D_i$")
axes[0, 1].set_xlabel("generation")
axes[0, 1].set_ylabel("unique tokens in corpus")
axes[0, 1].grid(alpha=0.3)

axes[1, 0].plot(metrics_df["generation"], metrics_df["corpus_entropy"],
                "o-", lw=2)
axes[1, 0].set_title("Token entropy of $D_i$")
axes[1, 0].set_xlabel("generation")
axes[1, 0].set_ylabel("entropy (bits)")
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(metrics_df["generation"], metrics_df["corpus_distinct_1"],
                "o-", lw=2, label="distinct-1")
axes[1, 1].plot(metrics_df["generation"], metrics_df["corpus_distinct_2"],
                "s--", lw=2, label="distinct-2")
axes[1, 1].set_title("Distinct-n")
axes[1, 1].set_xlabel("generation")
axes[1, 1].set_ylabel("ratio")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plot_path = PLOTS_DIR / "collapse_trajectory.png"
plt.savefig(plot_path, dpi=130, bbox_inches="tight")
plt.show()
print("saved plot", plot_path)


## 9. Pack everything for download

In [ ]:
print("packing artifacts", ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for root in (MODELS_DIR, SAMPLES_DIR, PLOTS_DIR):
        for path in root.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(BASE))
    for f in (METRICS_PATH, CONFIG_PATH):
        if f.exists():
            zf.write(f, f.name)

size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"wrote {ZIP_PATH} ({size_mb:.1f} MB)")